# Ambiguity-aware residual FCGR → DCT / SVD → RBF-SVM

CGR-preserving classifier for the synthetic diploid mTOR cohort.

Instead of treating IUPAC heterozygotes as additional nucleotides on a decagon, this notebook maps them to probabilities over the canonical alphabet. For example, `R` contributes `0.5 A + 0.5 G`. It constructs a standard `2^k × 2^k` FCGR and then measures the absolute count-space displacement from each reference-gene FCGR. This suppresses invariant reference composition and amplifies the mutation-load difference between classes.

Three reducers are compared under the same nested evaluation:

1. **DCT only** — retain a low-frequency square from each residual-magnitude FCGR;
2. **SVD only** — truncated SVD of all residual-magnitude FCGR values;
3. **DCT + SVD** — low-frequency residual DCT followed by truncated SVD.

Every arm uses a tuned RBF-SVM. Labels: **0 = control**, **1 = patient**.

In [8]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from scipy.fft import dctn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

DATA_PATH = Path("data_and_cache/mTOR_data.pkl")
REFERENCE_FASTA = Path("data_and_cache/mtor_referans_31.fasta")

RANDOM_SEED = 42
S_VALUES = [10, 20, 30, 40, 50]
N_RUNS = 10  # Use 100 for the final study.
KMER_SIZE = 5  # FCGR is 32 × 32 per gene (4^5 = 1024 cells).
HELLINGER_TRANSFORM = True
FCGR_CACHE = Path(
    f"data_and_cache/mTOR_expected_fcgr_k{KMER_SIZE}_hellinger{int(HELLINGER_TRANSFORM)}.npz"
)

# Residual magnitude places the mutation-load signal in the DCT DC/low-frequency
# coefficients, so very small retained blocks are meaningful.
DCT_KEEP_VALUES = [1, 2, 4, 8]
SVD_COMPONENTS = [1, 2, 5, 10, 20]
RBF_SVM_C = [0.1, 1.0, 10.0, 100.0]
RBF_SVM_GAMMA = ["scale", 0.001, 0.01, 0.1, 1.0]

np.random.seed(RANDOM_SEED)

In [9]:
def read_fasta(path):
    records = []
    name = None
    chunks = []
    with Path(path).open("r", encoding="utf-8", errors="ignore") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if name is not None:
                    records.append((name, "".join(chunks).upper()))
                name = line[1:].split()[0]
                chunks = []
            else:
                chunks.append("".join(char for char in line if char.isalpha()))
    if name is not None:
        records.append((name, "".join(chunks).upper()))
    return records


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} is missing. Run new_productıon_of_mTOR.ipynb first."
    )

with DATA_PATH.open("rb") as handle:
    sequence_data = pickle.load(handle)

controls = sequence_data["controls"]
patients = sequence_data["patients"]
gene_names = list(sequence_data["gene_names"])
subjects = controls + patients
y = np.concatenate([
    np.zeros(len(controls), dtype=np.int8),
    np.ones(len(patients), dtype=np.int8),
])

fasta_records = read_fasta(REFERENCE_FASTA)
reference_names = [name for name, _ in fasta_records]
references = [sequence for _, sequence in fasta_records]
if reference_names != gene_names:
    raise ValueError("FASTA and pickle gene order differ; align them before FCGR extraction.")
for subject_index, subject in enumerate(subjects):
    if len(subject) != len(references):
        raise ValueError(f"Subject {subject_index} has the wrong number of genes.")
    for gene_index, (observed, reference) in enumerate(zip(subject, references)):
        if len(observed) != len(reference):
            raise ValueError(
                f"Length mismatch at subject {subject_index}, gene {gene_names[gene_index]}."
            )

N_GENES = len(gene_names)
FCGR_SIDE = 2**KMER_SIZE
FEATURES_PER_GENE = 4**KMER_SIZE
print(
    f"Subjects={len(subjects)}, genes={N_GENES}, k={KMER_SIZE}, "
    f"FCGR={FCGR_SIDE}×{FCGR_SIDE}, raw features={N_GENES * FEATURES_PER_GENE:,}"
)

Subjects=800, genes=31, k=5, FCGR=32×32, raw features=31,744


## 1. Expected-count FCGR

For an IUPAC window, each compatible canonical k-mer receives its probability mass. A window with one heterozygous symbol therefore contributes `0.5` to each of two k-mers. Multiple ambiguities are marginalized independently; this is uncertainty-aware but does **not** infer phase.

Extraction starts from each reference gene's k-mer counts and recalculates only windows touched by a subject's non-reference symbols. This produces the same expected FCGR while avoiding a full Python loop over every base of every long gene.

In [10]:
IUPAC_OPTIONS = {
    "A": (0,), "C": (1,), "G": (2,), "T": (3,),
    "R": (0, 2), "Y": (1, 3), "M": (0, 1),
    "K": (2, 3), "S": (1, 2), "W": (0, 3),
}

OPTION_1 = np.full(256, -1, dtype=np.int8)
OPTION_2 = np.full(256, -1, dtype=np.int8)
for symbol, options in IUPAC_OPTIONS.items():
    OPTION_1[ord(symbol)] = options[0]
    if len(options) == 2:
        OPTION_2[ord(symbol)] = options[1]


def canonical_kmer_codes(sequence_bytes, k):
    bases = OPTION_1[sequence_bytes]
    if np.any(bases < 0) or np.any(OPTION_2[sequence_bytes] >= 0):
        raise ValueError("Reference FASTA must contain only canonical A/C/G/T bases.")
    n_windows = len(bases) - k + 1
    if n_windows <= 0:
        raise ValueError(f"Sequence length {len(bases)} is shorter than k={k}.")
    powers = (4 ** np.arange(k - 1, -1, -1)).astype(np.int64)
    codes = np.zeros(n_windows, dtype=np.int64)
    for offset, power in enumerate(powers):
        codes += bases[offset:offset + n_windows].astype(np.int64) * power
    return codes


def kmer_to_fcgr_permutation(k):
    """Map lexicographic A/C/G/T k-mer codes to square-CGR flattened cells."""
    n_kmers = 4**k
    side = 2**k
    corner_x = np.asarray([0, 0, 1, 1], dtype=np.int64)
    corner_y = np.asarray([0, 1, 0, 1], dtype=np.int64)
    permutation = np.empty(n_kmers, dtype=np.int64)

    for code in range(n_kmers):
        remainder = code
        digits = np.empty(k, dtype=np.int8)
        for position in range(k - 1, -1, -1):
            digits[position] = remainder % 4
            remainder //= 4
        x = sum(int(corner_x[base]) << position for position, base in enumerate(digits))
        y_coord = sum(int(corner_y[base]) << position for position, base in enumerate(digits))
        permutation[code] = y_coord * side + x
    return permutation


FCGR_PERMUTATION = kmer_to_fcgr_permutation(KMER_SIZE)
REFERENCE_ARRAYS = [
    np.frombuffer(sequence.encode("ascii"), dtype=np.uint8)
    for sequence in references
]
REFERENCE_CODES = [
    canonical_kmer_codes(sequence, KMER_SIZE) for sequence in REFERENCE_ARRAYS
]
REFERENCE_COUNTS = [
    np.bincount(codes, minlength=FEATURES_PER_GENE).astype(np.float64)
    for codes in REFERENCE_CODES
]

assert np.array_equal(np.sort(FCGR_PERMUTATION), np.arange(FEATURES_PER_GENE))
print("Reference k-mer tables and FCGR permutation prepared.")

Reference k-mer tables and FCGR permutation prepared.


In [11]:
def expected_counts_for_windows(sequence_bytes, starts, k, n_kmers):
    """Sum expected lexicographic k-mer counts for selected window starts."""
    result = np.zeros(n_kmers, dtype=np.float64)
    if len(starts) == 0:
        return result

    offsets = np.arange(k, dtype=np.int64)
    symbols = sequence_bytes[starts[:, None] + offsets[None, :]]
    first = OPTION_1[symbols]
    second = OPTION_2[symbols]
    if np.any(first < 0):
        invalid = sorted(set(symbols[first < 0].tobytes().decode("ascii")))
        raise ValueError(f"Unsupported IUPAC symbol(s): {invalid}")

    powers = (4 ** np.arange(k - 1, -1, -1)).astype(np.int64)
    ambiguity_count = np.sum(second >= 0, axis=1)

    canonical_rows = ambiguity_count == 0
    if np.any(canonical_rows):
        codes = first[canonical_rows].astype(np.int64) @ powers
        result += np.bincount(codes, minlength=n_kmers)

    one_ambiguity_rows = ambiguity_count == 1
    if np.any(one_ambiguity_rows):
        first_one = first[one_ambiguity_rows].astype(np.int64)
        second_one = second[one_ambiguity_rows].astype(np.int64)
        codes_first = first_one @ powers
        ambiguity_position = np.argmax(second_one >= 0, axis=1)
        row_index = np.arange(len(codes_first))
        codes_second = codes_first + (
            second_one[row_index, ambiguity_position]
            - first_one[row_index, ambiguity_position]
        ) * powers[ambiguity_position]
        result += 0.5 * np.bincount(codes_first, minlength=n_kmers)
        result += 0.5 * np.bincount(codes_second, minlength=n_kmers)

    multiple_rows = ambiguity_count >= 2
    if np.any(multiple_rows):
        first_multi = first[multiple_rows].astype(np.int64)
        second_multi = second[multiple_rows].astype(np.int64)
        # Enumerating 2^k choices duplicates canonical positions. Giving every
        # choice weight 1/2^k makes those duplicates sum to the correct 1/2^h
        # weight for a window containing h heterozygous positions.
        choice_weight = 1.0 / (2**k)
        for choice in range(2**k):
            choose_second = ((choice >> np.arange(k)) & 1).astype(bool)
            selected = np.where(
                choose_second[None, :] & (second_multi >= 0),
                second_multi,
                first_multi,
            )
            codes = selected @ powers
            result += choice_weight * np.bincount(codes, minlength=n_kmers)

    return result


def expected_gene_fcgr(
    observed,
    reference_array,
    reference_codes,
    reference_counts,
    k,
):
    sequence_bytes = np.frombuffer(observed.encode("ascii"), dtype=np.uint8)
    difference_positions = np.flatnonzero(sequence_bytes != reference_array)
    counts = reference_counts.copy()

    if difference_positions.size:
        affected_starts = (
            difference_positions[:, None] - np.arange(k, dtype=np.int64)[None, :]
        ).ravel()
        max_start = len(sequence_bytes) - k
        affected_starts = np.unique(
            affected_starts[(affected_starts >= 0) & (affected_starts <= max_start)]
        )
        counts -= np.bincount(
            reference_codes[affected_starts], minlength=len(reference_counts)
        )
        counts += expected_counts_for_windows(
            sequence_bytes, affected_starts, k, len(reference_counts)
        )

    n_windows = len(sequence_bytes) - k + 1
    if not np.isclose(counts.sum(), n_windows, atol=1e-6):
        raise AssertionError(
            f"Expected k-mer mass {counts.sum()} does not equal {n_windows} windows."
        )
    frequencies = np.maximum(counts / n_windows, 0.0)
    fcgr = np.empty_like(frequencies)
    fcgr[FCGR_PERMUTATION] = frequencies
    if HELLINGER_TRANSFORM:
        fcgr = np.sqrt(fcgr)
    return fcgr.astype(np.float32)


def extract_expected_fcgr(subjects):
    X = np.empty(
        (len(subjects), N_GENES * FEATURES_PER_GENE), dtype=np.float32
    )
    for subject_index, subject in enumerate(subjects):
        for gene_index, observed in enumerate(subject):
            start = gene_index * FEATURES_PER_GENE
            stop = start + FEATURES_PER_GENE
            X[subject_index, start:stop] = expected_gene_fcgr(
                observed,
                REFERENCE_ARRAYS[gene_index],
                REFERENCE_CODES[gene_index],
                REFERENCE_COUNTS[gene_index],
                KMER_SIZE,
            )
        if (subject_index + 1) % 10 == 0 or subject_index + 1 == len(subjects):
            print(f"Expected FCGR extraction: {subject_index + 1}/{len(subjects)}")
    return X


if FCGR_CACHE.exists():
    cached = np.load(FCGR_CACHE, allow_pickle=False)
    if (
        int(cached["kmer_size"]) != KMER_SIZE
        or list(cached["gene_names"]) != gene_names
        or bool(cached["hellinger_transform"]) != HELLINGER_TRANSFORM
    ):
        raise ValueError("FCGR cache settings do not match this notebook; delete and rebuild it.")
    X_fcgr = cached["X"]
    y_cached = cached["y"]
    if not np.array_equal(y_cached, y):
        raise ValueError("FCGR cache labels do not match the loaded dataset.")
    print(f"Loaded cached expected FCGR features: {X_fcgr.shape}")
else:
    X_fcgr = extract_expected_fcgr(subjects)
    FCGR_CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        FCGR_CACHE,
        X=X_fcgr,
        y=y,
        gene_names=np.asarray(gene_names),
        kmer_size=np.asarray(KMER_SIZE),
        hellinger_transform=np.asarray(HELLINGER_TRANSFORM),
    )
    print(f"Cached expected FCGR features: {X_fcgr.shape}")

if X_fcgr.shape != (len(y), N_GENES * FEATURES_PER_GENE):
    raise ValueError(f"Unexpected FCGR feature shape: {X_fcgr.shape}")

Loaded cached expected FCGR features: (800, 31744)


## 2. Reference-residual magnitude FCGR

The full expected FCGR is dominated by the common reference sequence, while each mutation only redistributes a small amount of k-mer mass. Moreover, signed FCGR differences sum to zero, so their DCT DC coefficient cannot represent how much sequence changed.

For classification, we therefore subtract each gene's reference FCGR in **count space** and take the absolute value:

`residual magnitude = |subject expected k-mer counts − reference k-mer counts|`

This remains an FCGR-derived image, but its intensity directly measures sequence change. Its DCT DC coefficient is proportional to total FCGR displacement, which deliberately retains the burden-separable signal in this synthetic dataset.

In [12]:
def make_reference_residual_magnitude(X_fcgr):
    reference_fcgr = np.empty((N_GENES, FEATURES_PER_GENE), dtype=np.float32)
    windows_per_gene = np.empty(N_GENES, dtype=np.float32)

    for gene_index, (reference_counts, reference_sequence) in enumerate(
        zip(REFERENCE_COUNTS, references)
    ):
        n_windows = len(reference_sequence) - KMER_SIZE + 1
        reference_frequency = reference_counts / n_windows
        ordered_reference = np.empty(FEATURES_PER_GENE, dtype=np.float64)
        ordered_reference[FCGR_PERMUTATION] = reference_frequency
        reference_fcgr[gene_index] = ordered_reference.astype(np.float32)
        windows_per_gene[gene_index] = n_windows

    subject_frequency = np.asarray(X_fcgr, dtype=np.float32).reshape(
        -1, N_GENES, FEATURES_PER_GENE
    )
    if HELLINGER_TRANSFORM:
        # Recover frequencies exactly enough from the cached square-root values.
        subject_frequency = np.square(subject_frequency, dtype=np.float32)
    else:
        subject_frequency = subject_frequency.copy()

    # Reuse the allocated array to keep peak memory reasonable.
    subject_frequency -= reference_fcgr[None, :, :]
    subject_frequency *= windows_per_gene[None, :, None]
    np.abs(subject_frequency, out=subject_frequency)
    return subject_frequency.reshape(len(X_fcgr), -1)


X_fcgr_residual = make_reference_residual_magnitude(X_fcgr)

# The first value is a useful diagnostic, not an extra classifier feature: it is
# the total amount of expected k-mer mass displaced from the reference.
residual_load = X_fcgr_residual.sum(axis=1)
print(
    f"Residual FCGR shape={X_fcgr_residual.shape}; "
    f"mean load control={residual_load[y == 0].mean():.1f}, "
    f"patient={residual_load[y == 1].mean():.1f}"
)
if residual_load[y == 1].mean() <= residual_load[y == 0].mean():
    raise AssertionError("Patient residual load was expected to exceed control load.")

Residual FCGR shape=(800, 31744); mean load control=27365.3, patient=28110.6


In [13]:
class PerGeneDCT(BaseEstimator, TransformerMixin):
    def __init__(self, n_genes=31, side=32, keep=8):
        self.n_genes = n_genes
        self.side = side
        self.keep = keep

    def fit(self, X, y=None):
        expected_features = self.n_genes * self.side * self.side
        if X.shape[1] != expected_features:
            raise ValueError(f"Expected {expected_features} FCGR values, got {X.shape[1]}.")
        if not 1 <= int(self.keep) <= self.side:
            raise ValueError(f"keep must be between 1 and {self.side}.")
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        if X.shape[1] != self.n_features_in_:
            raise ValueError(f"Expected {self.n_features_in_} features, got {X.shape[1]}.")
        images = np.asarray(X, dtype=np.float32).reshape(
            -1, self.n_genes, self.side, self.side
        )
        coefficients = dctn(images, axes=(-2, -1), norm="ortho")
        keep = int(self.keep)
        return coefficients[:, :, :keep, :keep].reshape(len(images), -1)


class AdaptiveTruncatedSVD(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=20, random_state=42, n_iter=7):
        self.n_components = n_components
        self.random_state = random_state
        self.n_iter = n_iter

    def fit(self, X, y=None):
        n_components = min(
            int(self.n_components),
            max(1, X.shape[0] - 1),
            max(1, X.shape[1] - 1),
        )
        self.n_components_ = n_components
        self.model_ = TruncatedSVD(
            n_components=n_components,
            algorithm="randomized",
            n_iter=self.n_iter,
            random_state=self.random_state,
        )
        self.model_.fit(X)
        self.explained_variance_ratio_ = self.model_.explained_variance_ratio_
        return self

    def transform(self, X):
        return self.model_.transform(X)


# Shape smoke tests on the residual-magnitude FCGR representation.
_demo = X_fcgr_residual[:3]
assert PerGeneDCT(N_GENES, FCGR_SIDE, keep=4).fit_transform(_demo).shape == (
    3, N_GENES * 4 * 4
)
assert AdaptiveTruncatedSVD(2, RANDOM_SEED).fit_transform(_demo).shape == (3, 2)
print("DCT and SVD reducer smoke tests passed.")

DCT and SVD reducer smoke tests passed.


## 3. Compare residual DCT, residual SVD, and residual DCT + SVD

All three arms receive exactly the same reference-residual FCGR images and outer train/test subjects. Reducer dimensions and the RBF-SVM `C` and `gamma` values are selected in inner stratified CV. This restores the nonlinear classifier used by the CGR–EMPR pipeline. The untouched outer remainder is evaluated once per arm and run.

In [18]:
def inner_cv(y_train, seed):
    folds = min(5, int(np.bincount(y_train).min()))
    if folds < 2:
        raise ValueError("Each class needs at least two inner-CV training subjects.")
    return StratifiedKFold(folds, shuffle=True, random_state=seed)


def make_search(method, y_train, seed, class_weight=None):
    svm = SVC(kernel="rbf", class_weight=class_weight)

    if method == "DCT":
        pipeline = Pipeline([
            ("dct", PerGeneDCT(N_GENES, FCGR_SIDE)),
            ("scale", StandardScaler()),
            ("svm", svm),
        ])
        parameter_grid = {
            "dct__keep": DCT_KEEP_VALUES,
            "svm__C": RBF_SVM_C,
            "svm__gamma": RBF_SVM_GAMMA,
        }
    elif method == "SVD":
        pipeline = Pipeline([
            ("center", StandardScaler(with_std=False)),
            ("svd", AdaptiveTruncatedSVD(random_state=seed)),
            ("scale", StandardScaler()),
            ("svm", svm),
        ])
        parameter_grid = {
            "svd__n_components": SVD_COMPONENTS,
            "svm__C": RBF_SVM_C,
            "svm__gamma": RBF_SVM_GAMMA,
        }
    elif method == "DCT + SVD":
        pipeline = Pipeline([
            ("dct", PerGeneDCT(N_GENES, FCGR_SIDE)),
            ("center", StandardScaler(with_std=False)),
            ("svd", AdaptiveTruncatedSVD(random_state=seed)),
            ("scale", StandardScaler()),
            ("svm", svm),
        ])
        parameter_grid = {
            "dct__keep": DCT_KEEP_VALUES,
            "svd__n_components": SVD_COMPONENTS,
            "svm__C": RBF_SVM_C,
            "svm__gamma": RBF_SVM_GAMMA,
        }
    else:
        raise ValueError(f"Unknown method: {method}")

    return RandomizedSearchCV(
        pipeline,
        parameter_grid,
        n_iter=30,
        scoring="f1",
        cv=inner_cv(y_train, seed),
        random_state=seed,
        n_jobs=1,  # Prevent repeated copies of the dense FCGR matrix.
        refit=True,
        error_score="raise",
    )


def evaluate_model(model, X_test, y_test):
    prediction = model.predict(X_test)
    score = model.decision_function(X_test)
    return {
        "OA (%)": 100.0 * accuracy_score(y_test, prediction),
        "Balanced accuracy": balanced_accuracy_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "Recall": recall_score(y_test, prediction, zero_division=0),
        "F1": f1_score(y_test, prediction, zero_division=0),
        "MCC": matthews_corrcoef(y_test, prediction),
        "AUC": roc_auc_score(y_test, score),
    }


def run_balanced_experiments(
    X,
    y,
    s_values=S_VALUES,
    n_runs=N_RUNS,
    seed=RANDOM_SEED,
):
    methods = ("DCT", "SVD", "DCT + SVD")
    class_indices = {label: np.flatnonzero(y == label) for label in (0, 1)}
    records = []

    for s in s_values:
        if s >= min(map(len, class_indices.values())):
            raise ValueError(f"S={s} leaves no outer-test samples in at least one class.")
        rng = np.random.default_rng(seed + s)

        for run in range(n_runs):
            train_indices = np.concatenate([
                rng.choice(class_indices[0], s, replace=False),
                rng.choice(class_indices[1], s, replace=False),
            ])
            test_indices = np.setdiff1d(np.arange(len(y)), train_indices)
            run_seed = seed + 1000 * s + run

            for method in methods:
                search = make_search(method, y[train_indices], run_seed)
                search.fit(X[train_indices], y[train_indices])
                metrics = evaluate_model(
                    search.best_estimator_, X[test_indices], y[test_indices]
                )
                records.append({
                    "Method": method,
                    "S": s,
                    "Run": run + 1,
                    **metrics,
                    **search.best_params_,
                })

        print(f"S={s}: {n_runs} outer runs × {len(methods)} reducers completed")

    return pd.DataFrame(records)


METRICS = [
    "OA (%)", "Balanced accuracy", "Precision", "Recall", "F1", "MCC", "AUC"
]


def summarize_runs(records):
    return (
        records.groupby(["Method", "S"])[METRICS]
        .agg(["mean", "std"])
        .round(4)
    )

In [15]:
fcgr_runs = run_balanced_experiments(X_fcgr_residual, y)
fcgr_summary = summarize_runs(fcgr_runs)
fcgr_summary

S=10: 10 outer runs × 3 reducers completed
S=20: 10 outer runs × 3 reducers completed
S=30: 10 outer runs × 3 reducers completed
S=40: 10 outer runs × 3 reducers completed
S=50: 10 outer runs × 3 reducers completed


OA (%)         Balanced accuracy         Precision          \
                 mean     std              mean     std      mean     std   
Method    S                                                                 
DCT       10  92.1282  4.7730            0.9213  0.0477    0.9517  0.0404   
          20  94.4474  3.1814            0.9445  0.0318    0.9565  0.0429   
          30  95.9324  1.6222            0.9593  0.0162    0.9618  0.0296   
          40  96.9861  1.1640            0.9699  0.0116    0.9775  0.0105   
          50  97.2714  0.5322            0.9727  0.0053    0.9724  0.0084   
DCT + SVD 10  84.2692  7.4538            0.8427  0.0745    0.8478  0.0928   
          20  90.2237  3.7674            0.9022  0.0377    0.9137  0.0578   
          30  89.9595  4.3566            0.8996  0.0436    0.9025  0.0534   
          40  91.4722  2.7841            0.9147  0.0278    0.9104  0.0329   
          50  92.4286  2.7063            0.9243  0.0271    0.9241  0.0425   
SVD       10  53.5256  4.4330            0.5353  0.0443    0.5013  0.1931   
          20  51.8684  3.1370            0.5187  0.0314    0.3680  0.2555   
          30  53.6351  2.5888            0.5364  0.0259    0.5162  0.1883   
          40  51.5000  2.8740            0.5150  0.0287    0.4955  0.1938   
          50  52.4571  4.2367            0.5246  0.0424    0.5467  0.0948   

              Recall              F1             MCC             AUC          
                mean     std    mean     std    mean     std    mean     std  
Method    S                                                                   
DCT       10  0.8887  0.0818  0.9172  0.0535  0.8469  0.0899  0.9764  0.0231  
          20  0.9329  0.0377  0.9439  0.0317  0.8903  0.0629  0.9863  0.0114  
          30  0.9578  0.0272  0.9593  0.0161  0.9196  0.0318  0.9921  0.0052  
          40  0.9619  0.0192  0.9696  0.0119  0.9400  0.0231  0.9940  0.0032  
          50  0.9731  0.0090  0.9727  0.0053  0.9455  0.0106  0.9946  0.0021  
DCT + SVD 10  0.8426  0.0605  0.8440  0.0706  0.6873  0.1494  0.9158  0.0639  
          20  0.8921  0.0398  0.9017  0.0361  0.8066  0.0756  0.9659  0.0293  
          30  0.8978  0.0403  0.8997  0.0424  0.7999  0.0875  0.9615  0.0336  
          40  0.9208  0.0315  0.9153  0.0274  0.8301  0.0552  0.9721  0.0146  
          50  0.9271  0.0360  0.9247  0.0257  0.8503  0.0520  0.9786  0.0155  
SVD       10  0.5203  0.3709  0.4605  0.2261  0.0757  0.0968  0.5680  0.0601  
          20  0.3924  0.4418  0.3110  0.3258  0.0386  0.0747  0.5650  0.0556  
          30  0.4227  0.3245  0.4143  0.2123  0.0892  0.0499  0.5709  0.0285  
          40  0.4989  0.3747  0.4283  0.2444  0.0463  0.0635  0.5548  0.0448  
          50  0.4603  0.3296  0.4246  0.2314  0.0529  0.0958  0.5549  0.0660

## 4. Imbalanced 1:4 experiment

The imbalanced pool contains all 400 controls and a fixed, reproducibly sampled subset of 100 patients. For each `S`, the training set contains `4S` controls and `S` patients; every remaining pool subject is used for outer testing.

The same outer split is shared by DCT, SVD, and DCT + SVD. RBF-SVM training uses `class_weight="balanced"`, while reducer and classifier hyperparameters remain confined to inner stratified cross-validation.

In [20]:
def make_imbalanced_pool(X, y, n_patients=100, seed=RANDOM_SEED):
    control_indices = np.flatnonzero(y == 0)
    patient_indices = np.flatnonzero(y == 1)
    if len(control_indices) < 4 * n_patients or len(patient_indices) < n_patients:
        raise ValueError("Not enough subjects to construct the requested 1:4 pool.")

    rng = np.random.default_rng(seed)
    selected_indices = np.concatenate([
        control_indices,
        rng.choice(patient_indices, n_patients, replace=False),
    ])
    return X[selected_indices], y[selected_indices], selected_indices


def run_imbalanced_experiments(
    X,
    y,
    s_values=S_VALUES,
    n_runs=N_RUNS,
    n_patients=100,
    seed=RANDOM_SEED,
):
    methods = ("DCT", "SVD", "DCT + SVD")
    pool_X, pool_y, pool_indices = make_imbalanced_pool(
        X, y, n_patients=n_patients, seed=seed
    )
    class_indices = {label: np.flatnonzero(pool_y == label) for label in (0, 1)}
    records = []

    for s in s_values:
        n_control_train = 4 * s
        if s >= len(class_indices[1]) or n_control_train >= len(class_indices[0]):
            raise ValueError(f"S={s} leaves no outer-test samples in at least one class.")
        rng = np.random.default_rng(seed + 10_000 + s)

        for run in range(n_runs):
            train_indices = np.concatenate([
                rng.choice(class_indices[0], n_control_train, replace=False),
                rng.choice(class_indices[1], s, replace=False),
            ])
            test_indices = np.setdiff1d(np.arange(len(pool_y)), train_indices)
            run_seed = seed + 20_000 + 1000 * s + run

            for method in methods:
                search = make_search(
                    method,
                    pool_y[train_indices],
                    run_seed,
                    class_weight="balanced",
                )
                search.fit(pool_X[train_indices], pool_y[train_indices])
                metrics = evaluate_model(
                    search.best_estimator_, pool_X[test_indices], pool_y[test_indices]
                )
                records.append({
                    "Method": method,
                    "S": s,
                    "Run": run + 1,
                    "Train controls": n_control_train,
                    "Train patients": s,
                    "Test controls": int(np.sum(pool_y[test_indices] == 0)),
                    "Test patients": int(np.sum(pool_y[test_indices] == 1)),
                    **metrics,
                    **search.best_params_,
                })

        print(
            f"Imbalanced S={s} ({n_control_train} controls, {s} patients): "
            f"{n_runs} outer runs × {len(methods)} reducers completed"
        )

    return pd.DataFrame(records), pool_indices

In [21]:
imbalanced_runs, imbalanced_pool_indices = run_imbalanced_experiments(
    X_fcgr_residual, y
)
imbalanced_summary = summarize_runs(imbalanced_runs)
imbalanced_summary

Imbalanced S=10 (40 controls, 10 patients): 10 outer runs × 3 reducers completed
Imbalanced S=20 (80 controls, 20 patients): 10 outer runs × 3 reducers completed
Imbalanced S=30 (120 controls, 30 patients): 10 outer runs × 3 reducers completed
Imbalanced S=40 (160 controls, 40 patients): 10 outer runs × 3 reducers completed
Imbalanced S=50 (200 controls, 50 patients): 10 outer runs × 3 reducers completed


OA (%)          Balanced accuracy         Precision          \
                 mean      std              mean     std      mean     std   
Method    S                                                                  
DCT       10  96.7556   1.8029            0.9489  0.0472    0.9254  0.0539   
          20  97.5000   0.7906            0.9614  0.0225    0.9396  0.0425   
          30  98.3429   0.6431            0.9762  0.0109    0.9547  0.0325   
          40  98.0667   0.4098            0.9667  0.0143    0.9601  0.0199   
          50  98.1600   0.5719            0.9645  0.0149    0.9713  0.0138   
DCT + SVD 10  90.2444   3.5299            0.8736  0.0480    0.7407  0.1022   
          20  92.3750   3.6558            0.9120  0.0176    0.7988  0.1462   
          30  92.0857   2.1969            0.9130  0.0371    0.7619  0.0851   
          40  92.9000   2.0187            0.9206  0.0484    0.7794  0.0486   
          50  93.2800   2.8209            0.9130  0.0394    0.8148  0.0955   
SVD       10  51.1111  25.7520            0.4969  0.0321    0.1425  0.1017   
          20  52.9750  21.0421            0.5266  0.0524    0.2463  0.0631   
          30  54.5429  14.3367            0.5305  0.0374    0.2299  0.0319   
          40  46.3000  14.7493            0.5125  0.0452    0.2130  0.0295   
          50  59.6800  11.7224            0.5470  0.0306    0.2456  0.0299   

              Recall              F1             MCC             AUC          
                mean     std    mean     std    mean     std    mean     std  
Method    S                                                                   
DCT       10  0.9178  0.1020  0.9168  0.0540  0.9003  0.0586  0.9943  0.0078  
          20  0.9388  0.0525  0.9373  0.0211  0.9231  0.0244  0.9974  0.0023  
          30  0.9643  0.0245  0.9589  0.0155  0.9490  0.0194  0.9980  0.0022  
          40  0.9433  0.0326  0.9511  0.0112  0.9395  0.0135  0.9976  0.0015  
          50  0.9360  0.0310  0.9530  0.0154  0.9420  0.0184  0.9969  0.0017  
DCT + SVD 10  0.8256  0.1053  0.7737  0.0701  0.7200  0.0878  0.9559  0.0333  
          20  0.8925  0.0703  0.8308  0.0594  0.7947  0.0716  0.9806  0.0100  
          30  0.9000  0.0819  0.8202  0.0467  0.7785  0.0610  0.9752  0.0182  
          40  0.9067  0.1043  0.8345  0.0545  0.7965  0.0666  0.9773  0.0193  
          50  0.8800  0.0794  0.8411  0.0585  0.8037  0.0744  0.9802  0.0149  
SVD       10  0.4733  0.4081  0.2093  0.1478 -0.0157  0.0631  0.5136  0.0424  
          20  0.5212  0.2662  0.3030  0.0434  0.0481  0.1014  0.5391  0.0615  
          30  0.5057  0.2636  0.2827  0.0881  0.0560  0.0607  0.5487  0.0555  
          40  0.5950  0.1918  0.3043  0.0384  0.0177  0.0773  0.5255  0.0661  
          50  0.4640  0.1748  0.3103  0.0423  0.0813  0.0503  0.5756  0.0426

### Interpretation checklist

- Compare all three arms on paired outer splits; do not compare their best individual runs.
- The representation intentionally exposes the dataset's mutation-load signal through reference-residual FCGR magnitude.
- `DCT keep=1` retains one DC coefficient per gene; this is the most compressed FCGR-derived load representation.
- SVD and all RBF-SVM hyperparameters are selected inside training CV, so the outer test set remains untouched.
- In the 1:4 experiment, prioritize balanced accuracy, patient recall, F1, MCC, and AUC over ordinary accuracy.
- The 100-patient imbalanced pool is fixed by `RANDOM_SEED`; change the seed or repeat with multiple pools to measure subset sensitivity.
- Expected FCGR marginalizes unphased heterozygotes independently. It must not be described as phased haplotype reconstruction.
- If residual DCT is strong but residual SVD is weak, report that supervised classification benefits from preserving explicit per-gene residual energy rather than forcing an unsupervised global low-rank subspace.